Comenzamos importando las librerias requeridas

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import seaborn.objects as so
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.cm as cm
import sqlite3
from formulaic import Formula
from sklearn import linear_model
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import train_test_split

# - Procesamiento de datos

Leemos los datos de full_data.csv

In [ ]:
data = pd.read_csv('full_data.csv')

Mostramos una parte de los datos para comenzar a analizar

In [ ]:
data.head()

## Ejercicio 1
Seleccionamos un subconjunto de las columnas. Para esto definimos una lista con los nombres de las columnas que usaremos.

In [ ]:
columnas = [
    'station_id',
    'num_bikes_available',
    'num_bikes_disabled',
    'num_docks_available',
    'num_docks_disabled',
    'Date',
    'hora',
    'dia'
]
datos_limpios = data[columnas]
datos_limpios.head()

## Ejercicio 2
Usamos un diccionario para convertir los nombres de las columnas a español

In [ ]:
columnas_a_espanol = {
    'station_id': 'estacion',
    'num_bikes_available': 'bicis_disponibles',
    'num_bikes_disabled': 'bicis_rotas',
    'num_docks_available': 'puertos_disponibles',
    'num_docks_disabled': 'puertos_rotos',
    'Date': 'fecha',
    'hora': 'horario',
    'dia': 'dia_semana'
}
datos_limpios = datos_limpios.rename(columns = columnas_a_espanol)
datos_limpios.head()

In [ ]:
datos_limpios['dia_semana'].unique()

## Ejercicio 3
Usamos un diccionario para convertir los dias de la semana a español

In [ ]:
dias_a_espaniol = {
    'Monday': 'Lunes',
    'Tuesday': 'Martes',
    'Wednesday': 'Miercoles',
    'Thursday': 'Jueves',
    'Friday': 'Viernes',
    'Saturday': 'Sabado',
    'Sunday': 'Domingo'
}
datos_limpios['dia_semana'] = datos_limpios['dia_semana'].map(dias_a_espaniol)

In [ ]:
datos_limpios.head()

## Ejercicio 4

Usamos info para ver que tipo de variables tenemos

Podemos ver que tenemos 5 variables de tipo entero, y tres de tipo *object* (strings)

In [ ]:
datos_limpios.info()

## Ejercicio 5
Hay datos faltantes? Usamos isna y sum

In [ ]:
datos_limpios.isna().sum()

Podemos ver que la base de datos no tiene datos faltantes.

## Ejercicio 6

In [ ]:
anio = datos_limpios['fecha'].map(lambda fecha: fecha.split('-')[0]).astype(int)
mes = datos_limpios['fecha'].map(lambda fecha: fecha.split('-')[1]).astype(int)
dia = datos_limpios['fecha'].map(lambda fecha: fecha.split('-')[2]).astype(int)
hora = datos_limpios['horario'].map(lambda horario: horario.split(':')[0]).astype(int)
hora.head()

In [ ]:
# Guardo las columnas en el dataframe datos_limpios
datos_limpios['anio'] = anio
datos_limpios['mes'] = mes
datos_limpios['dia'] = dia
datos_limpios['hora'] = hora
datos_limpios.head()

## Ejercicio 7

In [ ]:
# Usando datetime
# datos_limpios['fecha'] = pd.to_datetime(datos_limpios['fecha'])
# datos_limpios['fecha']

## Ejercicio 8

In [ ]:
def determinar_estacion_anio(fecha):
    estaciones = fecha.apply(
    lambda row:
        'Otoño' if (row.month == 3 and row.day >= 20) or (3 < row.month < 6) or (row.month == 6 and row.day <= 21)
        else ('Invierno' if (row.month == 6 and row.day >= 21) or (6 < row.month < 9) or (row.month == 9 and row.day <= 22)
             else (
                 'Primavera' if (row.month == 9 and row.day >= 22) or (9 < row.month < 12) or (row.month == 12 and row.day <= 21)
                 else 'Verano'
             ))

    )

    return estaciones

In [ ]:
fecha = pd.to_datetime(datos_limpios['fecha'])
fecha.head()

In [ ]:
estaciones = determinar_estacion_anio(fecha)
estaciones

In [ ]:
estaciones.value_counts()

In [ ]:
datos_limpios['estaciones'] = estaciones

In [ ]:
datos_limpios.head()

In [ ]:
datos_limpios['estaciones'].value_counts()

# - Análisis descriptivos

## Ejercicio 9
Creamos dataframe nuevo llamado "datos_agrupados" que contiene nuevas variables "cantidad_bicis_disponbles", "cantidad_bicis_rotas" y "cantidad_puertos_disponibles".

In [ ]:
datos_agrupados = datos_limpios.groupby(["fecha", "hora"]).agg({
    "bicis_disponibles": "sum",
    "bicis_rotas": "sum",
    "puertos_disponibles": "sum",
    "dia_semana": "first",
    "dia": "first",
    "mes": "first",
    "anio": "first",
    "estaciones": "first"
}).reset_index()

datos_agrupados = datos_agrupados.rename(columns={
    "bicis_disponibles": "cantidad_bicis_disponibles",
    "bicis_rotas": "cantidad_bicis_rotas",
    "puertos_disponibles": "cantidad_puertos_disponibles"
})

In [ ]:
datos_agrupados

In [ ]:
datos_agrupados.info()

## Ejercicio 10
En que estacion suele haber mas "cantidad_bicis_disponbles" y "cantidad_puertos_disponibles".

In [ ]:
sns.boxplot(data = datos_agrupados, x = "estaciones", y = "cantidad_bicis_disponibles")
plt.show()

Se ve que la mediana mas alta es la de Otoño, entonces esa es la estacion donde suele haber mas bicis disponibles en el año.

In [ ]:
sns.boxplot(data = datos_agrupados, x = "estaciones", y = "cantidad_puertos_disponibles")
plt.show()

Se ve que la mediana mas alta es la de Primavera, entonces esa es la estacion donde suele haber mas puertos disponibles en el año.

## Ejercicio 11
Mostramos como cambia la cantidad de bicis y puertos disponibles promedio a lo largo del dia segun dia y horario.
Creamos "datos_agrupados_para_graficar_2" que es igual a "datos_agrupados_para_graficar" pero con el "horario_24hs"

In [ ]:
def determinar_hora(horario):
    horas = []
    for hora in horario:
        horas.append(hora.split(':')[0])
    return horas

# horario_24hs = determinar_hora(datos_agrupados_para_graficar["horario"])
# datos_agrupados_para_graficar_2 = datos_agrupados_para_graficar.copy()
# datos_agrupados_para_graficar_2["horario_24hs"] = horario_24hs
# datos_agrupados_para_graficar_2.head()

Agrupamos segun "dia_semana" y "horario_24hs".
Despues calculamos el promedio de "cantidad_bicis_disponibles", "cantidad_bicis_rotas" y "cantidad_puertos_disponibles".


In [ ]:
datos_agrupados_para_graficar_2 = datos_agrupados.groupby(["dia_semana", "hora"]).agg({
    "cantidad_bicis_disponibles": "mean",
    "cantidad_bicis_rotas": "mean",
    "cantidad_puertos_disponibles": "mean",
}).reset_index()
datos_agrupados.head()

In [ ]:
(
    so.Plot(data = datos_agrupados_para_graficar_2, x = "hora", y = "cantidad_bicis_disponibles")
    .add(so.Line(), color = "dia_semana")
)

In [ ]:
(
    so.Plot(data = datos_agrupados_para_graficar_2, x = "hora", y = "cantidad_puertos_disponibles")
    .add(so.Line(), color = "dia_semana")
)

## Ejercicio 12
En los graficos se ve que entre las 21 a 23 hay un comportamiento raro todos los dias.
Se ve que a las 21 se siguen usando las bicis, a las 22 muchas personas dejan las bicis y a las 23 muchas bicis estan siendo usadas.
Como es imposible que el comportamiento humano sea tan perfectamente igual todos los dias y en el mismo horario puede ser que haya algun error como la falta de datos entre las 21 y 23 hs.

# - Analisis exploratorio

## Ejercicio 13

In [ ]:
datos_limpios.head(5)

Agrupamos por estacion (420 y 464). Queremos ver que diferencias hay entre una y otra.

In [ ]:
datos_agrupados_estacion = (
    datos_limpios
    .groupby(["hora", "estacion", "mes"])
    .agg({
        'bicis_disponibles': 'mean',
        'puertos_disponibles': 'mean'
    })
).reset_index()
datos_agrupados_estacion.head()

Graficamos, para cada mes, la cantidad promedio de bicis y puertos disponibles por hora de cada estacion

In [ ]:
g = sns.FacetGrid(
    datos_agrupados_estacion,
    col="mes",
    hue="estacion",
    col_wrap=4
)
g.map(
    sns.lineplot,
    "hora",
    "bicis_disponibles",
)
g.add_legend(title="Estación")
g.fig.subplots_adjust(top=0.9)
g.fig.suptitle('Bicis disponibles por mes', fontsize=16);

In [ ]:
g = sns.FacetGrid(
    datos_agrupados_estacion,
    col="mes",
    hue="estacion",
    col_wrap=3
)
g.map(
    sns.lineplot,
    "hora",
    "puertos_disponibles",
)
g.add_legend(title="Estación")
g.fig.subplots_adjust(top=0.9)
g.fig.suptitle('Puertos disponibles por mes', fontsize=16);

A partir del conjunto de graficos, podemos concluir que tanto la cantidad promedio de bicis como de puertos disponibles en la estacion 464 es mayor que en la 420, implicando que esta estacion es mas frecuentada que la otra

Vamos a analizar tambien como afecta el clima durante el dia el uso de las bicicletas

In [ ]:
clima=pd.read_csv("clima.csv")
clima

In [ ]:
clima.info()

Pasamos ambos dataframes a sql para poder unir los datos de clima al resto de los datos con los que ya venimos trabajando

In [ ]:
con = sqlite3.connect(":memory:")

datos_limpios.to_sql("datos_agrupados", con, index=False, if_exists="replace")
clima.to_sql("clima", con, index=False, if_exists="replace")

In [ ]:
datos_con_clima = pd.read_sql_query("""
SELECT d.*, c.tavg as temp_promedio, c.prcp as lluvia
FROM datos_agrupados d
LEFT JOIN clima c
ON c.fecha = d.fecha
""", con)

datos_con_clima

Eliminamos las filas con valores nulos

In [ ]:
datos_con_clima= datos_con_clima.dropna()

In [ ]:
datos_con_clima.head()

Lo pasamos a sql para poder tratar los datos de forma mas comoda antes de graficar

In [ ]:
datos_con_clima.to_sql("datos_con_clima", con, index=False, if_exists="replace")

Queremos ver como afecta el uso de bicicletas la temperatura segun el mes del año, para eso agrupamos por mes y calculamos el promedio de las bicis y la temperatura para simplificar la visualizacion de la informacion

In [ ]:
datos_con_clima_mensual = pd.read_sql_query("""
SELECT
    AVG(bicis_disponibles) as bicis_disponibles,
    AVG(bicis_rotas) as bicis_rotas,
    AVG(puertos_disponibles) as puertos_disponibles,
    AVG(temp_promedio) as temp_promedio,
    MAX(estaciones) as estaciones,
    mes
FROM datos_con_clima
GROUP BY mes
""", con)
datos_con_clima_mensual

In [ ]:
(
    so.Plot(
        data = datos_con_clima_mensual, 
        x = "mes", 
        y = "puertos_disponibles", 
        color="temp_promedio"
    )
    .add(so.Bars())
    .label(title="Puertos disponibles y temperatura promedio por mes")
    .scale(color="coolwarm")
)

Agrupamos por estacion del año para que se vea mas clara la tendencia respecto de la temperatura

In [ ]:
clima_por_estacion = (
    datos_con_clima_mensual
    .groupby("estaciones")
    .agg({
    "temp_promedio": "mean",
    "puertos_disponibles": "mean",
})
)

(
    so.Plot(
        data = clima_por_estacion, 
        x = "estaciones", 
        y = "puertos_disponibles", 
        color="temp_promedio"
    )
    .add(so.Bars())
    .label(title="Puertos disponibles y temperatura promedio por estacion")
    .scale(color="coolwarm")
)

Podemos observar que en meses y estaciones mas frias, como Junio y Julio (Invierno), desciende la cantidad de puertos disponibles, es decir, hay menos bicicletas en uso.

Ahora queremos ver como las condiciones climaticas (lluvia) afectan al uso de bicis

In [ ]:
datos_con_lluvia_mensual = pd.read_sql_query("""
SELECT
    SUM(puertos_disponibles) as puertos_disponibles,
    SUM(bicis_disponibles) as bicis_disponibles,
    SUM(lluvia) as lluvia,
    MAX(estaciones) as estaciones,
    dia,
    mes
FROM datos_con_clima
GROUP BY dia, mes
""", con)
datos_con_lluvia_mensual.head()

Graficamos, para cada mes, puertos disponibles por dia para ver cuantas bicis estaban en uso segun la cantidad de lluvia.

In [ ]:
vmin = datos_con_lluvia_mensual["lluvia"].min()
vmax = datos_con_lluvia_mensual["lluvia"].max()
cmap = plt.get_cmap("rocket_r")
norm = mcolors.Normalize(vmin=vmin, vmax=vmax)

lluvia_values = datos_con_lluvia_mensual["lluvia"].unique()
palette = {v: cmap(norm(v)) for v in lluvia_values}

g = sns.FacetGrid(
    datos_con_lluvia_mensual,
    col="mes",
    hue="lluvia",
    palette=palette, 
    legend_out=True,
    col_wrap=4
)
g.map(
    sns.barplot,
    "dia",
    "puertos_disponibles",
    order=range(1, 32),
)
g.set(xticks=[0, 9, 19, 29, 32])

sm = cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])

g.fig.subplots_adjust(right=0.88)
cbar_ax = g.fig.add_axes([0.91, 0.15, 0.02, 0.7])
g.fig.colorbar(sm, cax=cbar_ax, label="lluvia")

A partir de estos graficos no podemos concluir que haya una relacion entre cantidad de lluvia y uso de bicis

# Regresion Lineal

## Ejercicio 15
Se quiere ajustar la cantidad de viajes con origen en la estacion 005 ("origen_5") - Plaza Italia en funcion de viajes originados en otras estaciones o con destino en distintas estaciones. Es decir, queremos hacer un modelo para ajustar la variable "origen_5" en funcion de otras variables del DataFrame

In [ ]:
viajes_diarios = pd.read_csv('viajes_diarios.csv')
viajes_diarios

In [ ]:
# Veamos si existen valores faltantes
viajes_diarios.isna().sum().sum()

Graficamos correlacion entre algunas variables para entender si existe alguna relacion evidente

In [ ]:
sns.pairplot(viajes_diarios[[
    "origen_5", 
    # "destino_175", 
    # "destino_516", 
    # "destino_14", 
    # "destino_202",
    # "destino_8",
    "origen_175", 
    "origen_516", 
    "origen_14", 
    "origen_202",
    "origen_8",
]])

A partir de los graficos de correlacion, se puede ver que hay una ligera relacion directa entre las variables origen_5 y destino

Por otro lado, observando los valores de la cantidad de viajes de cada variable (por ejemplo origen_5 va de 0 a 200, mientras que destino_7 va de 0 a 40 aproximadamente), podemos pensar que puertos con cantidad de viajes similares podrian estar relacionados

Calculemos cuantos viajes promedio tiene cada variable

In [ ]:
viajes_valor_medio = []
for c in viajes_diarios.select_dtypes(include='number').columns:
    if 'origen' in c or c == 'origen_5':
        viajes_valor_medio.append((viajes_diarios[c].mean(), c))

viajes_valor_medio.sort(reverse=True)
viajes_valor_medio[:15]

## Ejercicio 16
Elegimos 3 modelos de regresion. Elegimos las variables cuyo valor medio sea similar a origen_5

Las variables explicativas de cada modelo son:

Modelo 1: destino_175, destino_516, destino_14, destino_202, destino_8

Modelo 2: origen_175, origen_516, origen_14, origen_202, origen_8

Modelo 3: destino_175, origen_516, destino_14, origen_202, destino_8

## Ejercicio 17

Construimos las matrices X e y del modelo 1, ajustamos el modelo separando en 80-20

In [ ]:
# MODELO 1
modelo1_y, modelo1_X = (
    Formula('origen_5 ~ destino_175 + destino_516 + destino_14 + destino_202 + destino_8 - 1')
    .get_model_matrix(viajes_diarios)
)

In [ ]:
# Separamos en entrenamiento (train) y testeo (test). 
modelo1_X_train, modelo1_X_test, modelo1_y_train, modelo1_y_test = train_test_split(modelo1_X, modelo1_y, test_size=0.2, random_state=1)

In [ ]:
modelo1_X_train

In [ ]:
modelo1_X_test

In [ ]:
# Entrenamos el modelo utilizando los conjuntos de entrenamiento

modelo1 = linear_model.LinearRegression(fit_intercept = True)
modelo1.fit(modelo1_X_train, modelo1_y_train)

TEST

In [ ]:
modelo1_y_pred = modelo1.predict(modelo1_X_test)

# Calculando el R^2
modelo1_r2 = r2_score(modelo1_y_test, modelo1_y_pred)
print('R^2: ', modelo1_r2)

# Calculando la RECM
modelo1_ecm = mean_squared_error(modelo1_y_test, modelo1_y_pred)
print('Raiz cuadarada del ECM: ', np.sqrt(modelo1_ecm))

Construimos las matrices X e y del modelo 2, ajustamos el modelo separando en 80-20

In [ ]:
# MODELO 2
modelo2_y, modelo2_X = (
    Formula('origen_5 ~ origen_175 + origen_516 + origen_14 + origen_202 + origen_8 - 1')
    .get_model_matrix(viajes_diarios)
)

In [ ]:
# Separamos en entrenamiento (train) y testeo (test). 
modelo2_X_train, modelo2_X_test, modelo2_y_train, modelo2_y_test = train_test_split(modelo2_X, modelo2_y, test_size=0.2, random_state=1)

In [ ]:
modelo2_X_train

In [ ]:
modelo2_X_test

In [ ]:
# Entrenamos el modelo utilizando los conjuntos de entrenamiento

modelo2 = linear_model.LinearRegression(fit_intercept = True)
modelo2.fit(modelo2_X_train, modelo2_y_train)

TEST

In [ ]:
modelo2_y_pred = modelo2.predict(modelo2_X_test)

# Calculando el R^2
modelo2_r2 = r2_score(modelo2_y_test, modelo2_y_pred)
print('R^2: ', modelo2_r2)

# Calculando la RECM
modelo2_ecm = mean_squared_error(modelo2_y_test, modelo2_y_pred)
print('Raiz cuadarada del ECM: ', np.sqrt(modelo2_ecm))

Construimos las matrices X e y del modelo 3, ajustamos el modelo separando en 80-20

In [ ]:
# MODELO 3
modelo3_y, modelo3_X = (
    Formula('origen_5 ~ destino_175 + origen_516 + destino_14 + origen_202 + destino_8 - 1')
    .get_model_matrix(viajes_diarios)
)

In [ ]:
# Separamos en entrenamiento (train) y testeo (test). 
modelo3_X_train, modelo3_X_test, modelo3_y_train, modelo3_y_test = train_test_split(modelo3_X, modelo3_y, test_size=0.2, random_state=1)

In [ ]:
modelo3_X_train

In [ ]:
modelo3_X_test

In [ ]:
# Entrenamos el modelo utilizando los conjuntos de entrenamiento

modelo3 = linear_model.LinearRegression(fit_intercept = True)
modelo3.fit(modelo3_X_train, modelo3_y_train)

TEST

In [ ]:
modelo3_y_pred = modelo3.predict(modelo3_X_test)

# Calculando el R^2
modelo3_r2 = r2_score(modelo3_y_test, modelo3_y_pred)
print('R^2: ', modelo3_r2)

# Calculando la RECM
modelo3_ecm = mean_squared_error(modelo3_y_test, modelo3_y_pred)
print('Raiz cuadarada del ECM: ', np.sqrt(modelo3_ecm))

Hacemos las comparacion para seleccionar el mejor modelo:

In [ ]:
comparacion = {
    'Modelo': ['Modelo 1', 'Modelo 2', 'Modelo 3'],
    'R2': [modelo1_r2, modelo2_r2, modelo3_r2],
    'RECM': [np.sqrt(modelo1_ecm), np.sqrt(modelo2_ecm), np.sqrt(modelo3_ecm)]
}
df_comparacion = pd.DataFrame(comparacion)

df_comparacion

El mejor modelo es el que tiene R2 mas alto Y RECM mas bajo.

Entonces el mejor modelo es el Modelo 2

## Ejercicio 18
Entonces la formula final del Modelo 2 es:

$$ origen\_5 = \beta_0 + \beta_1 \cdot origen\_175 + \beta_2 \cdot origen\_516 + \beta_3 \cdot origen\_14 + \beta_4 \cdot origen\_202 + \beta_5 \cdot origen\_8 $$

origen_175 + origen_516 + origen_14 + origen_202 + origen_8

In [ ]:
modelo2.intercept_, modelo2.coef_

Formula final:
$$origen\_5 = 3.60 - 0.0099 \cdot origen\_175 + 0.269 \cdot origen\_516 + 0.2897 \cdot origen\_14 + 0.127 \cdot origen\_202 + 0.285 \cdot origen\_8$$
